In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json, math, shutil, random, re
from collections import deque
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup
from datasets import load_dataset
from sklearn.metrics import f1_score
from tqdm import tqdm

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [2]:
!pip install senticnet

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

VIZ_DIR = "viz_outputs"
os.makedirs(VIZ_DIR, exist_ok=True)

In [4]:
class RealSenticNetPrior:
    def __init__(self, emotion_names, fallback_json_path=None, bfs_depth=3, max_concepts_per_text=32):
        self.emotion_names = emotion_names
        self.label_concepts = [self._norm(x) for x in emotion_names]
        self.bfs_depth = bfs_depth
        self.max_concepts_per_text = max_concepts_per_text
        
        self.cache_sem = {}
        self.cache_has = {}
        self._cached_label_aff = None
        
        self.sn = None
        self.fallback = None
        
        try:
            from senticnet.senticnet import SenticNet
            self.sn = SenticNet()
            print("[INFO] ✓ SenticNet loaded")
        except Exception:
            try:
                from senticnet.senticnet import Senticnet
                self.sn = Senticnet()
                print("[INFO] ✓ SenticNet (alt) loaded")
            except Exception:
                self.sn = None
        
        if self.sn is None and fallback_json_path and os.path.exists(fallback_json_path):
            with open(fallback_json_path, "r", encoding="utf-8") as f:
                self.fallback = json.load(f)
        
        if self.sn is None and self.fallback is None:
            print("[WARN] SenticNet unavailable. Install: pip install -U senticnet")
    
    @staticmethod
    def _norm(s: str) -> str:
        s = s.lower().strip()
        s = re.sub(r"[^a-z0-9\s_]+", "", s)
        s = re.sub(r"\s+", "_", s)
        return s

    def get_adj_matrix(self):
        n = len(self.emotion_names)
        adj = np.eye(n, dtype=np.float32) # Self-loops
        idx = {self._norm(name): i for i, name in enumerate(self.emotion_names)}

        # 1. SenticNet BFS connections
        for i, name in enumerate(self.label_concepts):
            sems = self.get_semantics(name)
            for s in sems:
                if s in idx:
                    adj[i, idx[s]] = 1.0
    
    def get_semantics(self, c: str):
        c = self._norm(c)
        if c in self.cache_sem:
            return self.cache_sem[c]
        
        sems = []
        if self.sn is not None and hasattr(self.sn, "semantics"):
            try:
                sems = self.sn.semantics(c)
            except Exception:
                sems = []
        elif self.fallback is not None:
            sems = self.fallback.get(c, {}).get("semantics", [])
        
        sems = [self._norm(x) for x in sems if isinstance(x, str)]
        self.cache_sem[c] = sems
        return sems
    
    def has_concept(self, c: str) -> bool:
        c = self._norm(c)
        if c in self.cache_has:
            return self.cache_has[c]
        
        ok = False
        if self.sn is not None:
            try:
                _ = self.get_semantics(c)
                ok = True
            except Exception:
                ok = False
        elif self.fallback is not None:
            ok = c in self.fallback
        
        self.cache_has[c] = ok
        return ok
    
    def extract_concepts(self, text: str):
        t = text.lower()
        t = re.sub(r"http\S+|www\.\S+", " ", t)
        t = re.sub(r"[^a-z0-9\s]+", " ", t)
        toks = [x for x in t.split() if x]
        
        cands = []
        for n in (1, 2, 3):
            for i in range(len(toks) - n + 1):
                cands.append("_".join(toks[i:i+n]))
        
        cands = sorted(set(cands), key=lambda x: (-len(x), x))
        
        found = []
        for c in cands:
            if len(found) >= self.max_concepts_per_text:
                break
            if self.has_concept(c):
                found.append(self._norm(c))
        return found
    
    def compute_label_prior(self, text: str):
        if (self.sn is None) and (self.fallback is None):
            return np.zeros(len(self.emotion_names), dtype=np.float32)
        
        concepts = self.extract_concepts(text)
        if not concepts:
            return np.zeros(len(self.emotion_names), dtype=np.float32)
        
        label_to_idx = {c: i for i, c in enumerate(self.label_concepts)}
        scores = np.zeros(len(self.emotion_names), dtype=np.float32)
        
        for src in concepts:
            q = deque([(src, 0)])
            seen = {src}
            while q:
                node, d = q.popleft()
                if d > self.bfs_depth:
                    continue
                
                if node in label_to_idx:
                    scores[label_to_idx[node]] += 1.0 / (d + 1.0)
                
                if d == self.bfs_depth:
                    continue
                
                for nb in self.get_semantics(node):
                    if nb not in seen:
                        seen.add(nb)
                        q.append((nb, d + 1))
        
        s = scores.sum()
        if s > 0:
            scores = scores / (s + 1e-8)
        return scores
    
    def _get_sentics_4(self, concept: str):
        concept = self._norm(concept)
        
        if self.sn is not None and hasattr(self.sn, "sentics"):
            try:
                out = self.sn.sentics(concept)
                if isinstance(out, dict):
                    vals = [
                        out.get("pleasantness", 0.0),
                        out.get("attention", 0.0),
                        out.get("sensitivity", 0.0),
                        out.get("aptitude", 0.0),
                    ]
                else:
                    vals = list(out)[:4]
                vals = [float(v) for v in vals]
                if len(vals) == 4:
                    return vals
            except Exception:
                pass
        
        if self.fallback is not None:
            try:
                sentics = self.fallback.get(concept, {}).get("sentics", None)
                if sentics is not None and len(sentics) >= 4:
                    return [float(v) for v in sentics[:4]]
            except Exception:
                pass
        
        return [0.0, 0.0, 0.0, 0.0]
    
    def get_label_affective_scores(self):
        if self._cached_label_aff is not None:
            return self._cached_label_aff
        
        M = []
        for lab in self.label_concepts:
            M.append(self._get_sentics_4(lab))
        
        self._cached_label_aff = torch.tensor(M, dtype=torch.float32)
        return self._cached_label_aff

In [5]:
EKMAN_CLASSES = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
EKMAN_MAP = {
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    "disgust": "disgust",
    "fear": "fear", "nervousness": "fear",
    "joy": "joy", "amusement": "joy", "excitement": "joy", "love": "joy",
    "gratitude": "joy", "admiration": "joy", "approval": "joy", "optimism": "joy",
    "pride": "joy", "desire": "joy", "caring": "joy", "relief": "joy",
    "sadness": "sadness", "disappointment": "sadness", "grief": "sadness",
    "remorse": "sadness", "embarrassment": "sadness",
    "surprise": "surprise", "realization": "surprise", "confusion": "surprise",
    "curiosity": "surprise",
    "neutral": "neutral",
}

SUPEREMOTION_CLASSES = [
    "joy", "sadness", "anger",
    "fear", "love", "surprise", "neutral"
]

In [6]:
def build_ekman_index(emotion_names):
    ek_idx = {k: i for i, k in enumerate(EKMAN_CLASSES)}
    return {i: ek_idx[EKMAN_MAP.get(n, "neutral")] for i, n in enumerate(emotion_names)}

def make_ekman_targets(labels_28, emo_to_ek, num_ek=7):
    B = labels_28.size(0)
    labels_ek = torch.zeros(B, num_ek, device=labels_28.device, dtype=labels_28.dtype)
    for j in range(len(emo_to_ek)):   # was range(28)
        labels_ek[:, emo_to_ek[j]] = torch.maximum(labels_ek[:, emo_to_ek[j]], labels_28[:, j])
    return labels_ek

In [7]:
class GoEmotionsEkman7Dataset(Dataset):
    def __init__(self, data, tokenizer, max_len=96):
        print("Pre-tokenizing...")
        self.texts = [item["text"] for item in data]
        self.enc = tokenizer(
            self.texts,
            max_length=max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        ekman_idx = {e: i for i, e in enumerate(EKMAN_CLASSES)}
        emotion_names_28 = data.features["labels"].feature.names

        self.labels = []
        for item in data:
            vec = torch.zeros(7)
            for label_id in item["labels"]:
                fine = emotion_names_28[label_id]
                ek   = EKMAN_MAP.get(fine, "neutral")
                vec[ekman_idx[ek]] = 1.0
            self.labels.append(vec)
        self.labels = torch.stack(self.labels)

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return {
            "text": self.texts[i],
            "input_ids": self.enc["input_ids"][i],
            "attention_mask": self.enc["attention_mask"][i],
            "labels": self.labels[i],
        }

In [8]:
class AdaptiveGraphLayer(nn.Module):
    def __init__(self, dim, heads=4, max_k=10, drop=0.1):
        super().__init__()
        self.max_k = max_k
        self.attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(dim * 2, dim),
        )
        self.norm1 = nn.LayerNorm(dim)
        self.norm2 = nn.LayerNorm(dim)
        self.drop = nn.Dropout(drop)
    
    def forward(self, x, current_k=None):
        if current_k is None:
            current_k = self.max_k
        
        x_in = x.unsqueeze(0)
        _, attn_w = self.attn(x_in, x_in, x_in, need_weights=True, average_attn_weights=True)
        attn_w = attn_w.squeeze(0)
        
        k_use = min(current_k, attn_w.size(1))
        _, idx = torch.topk(attn_w, k_use, dim=1)
        mask = torch.zeros_like(attn_w).scatter_(1, idx, 1.0)
        attn_s = attn_w * mask
        attn_s = attn_s / (attn_s.sum(1, keepdim=True) + 1e-8)
        
        out = torch.matmul(attn_s.unsqueeze(0), x_in)
        x_mid = self.norm1(x_in + self.drop(out))
        x_out = self.norm2(x_mid + self.drop(self.ffn(x_mid))).squeeze(0)
        return x_out

In [9]:
class DESModule(nn.Module):

    def __init__(self, dim=768):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        # x is the sequence of hidden states from RoBERTa
        attn_out, _ = self.mha(x, x, x)
        return self.norm(x + attn_out)

In [10]:
class HybridSenticGNN(nn.Module):
    """
    DUAL-STREAM FUSION:
    - Stream 1: RoBERTa (formal/academic English)
    - Stream 2: Twitter-RoBERTa-Emotion (social/slang/informal)
    - Fusion Gate: Learnable combination
    - GNN: Operates on fused features
    """
    def __init__(
            self,
            base_model_name="roberta-base",
            social_model_name="cardiffnlp/twitter-roberta-base-emotion", # Use this as the 'X' backbone
            emotion_names=None,
            tokenizer=None,
            n_emo=28,
            n_ek=7,
            dim=384,
            n_layers=2,
            heads=4,
            max_k=10,
            drop=0.1,
            sentic=None,
            freeze_first_n_layers=2,
            knowledge_warmup_steps=1000,
            prior_scale=0.30,
            init_from_label_texts=True,
            use_social_stream=True,
        ):
            super().__init__()
            
            # --- Stream 1: Formal/Academic ---
            print(f"[1/2] Loading base encoder: {base_model_name}")
            self.encoder_base = AutoModel.from_pretrained(base_model_name)
            base_dim = self.encoder_base.config.hidden_size
            
            # --- Stream 2: Social/Informal (The EmoBERTa-X Path) ---
            self.use_social_stream = use_social_stream
            if self.use_social_stream:
                print(f"[2/2] Loading social encoder: {social_model_name}")
                try:
                    self.encoder_social = AutoModel.from_pretrained(social_model_name)
                    social_dim = self.encoder_social.config.hidden_size
    
                    # ADDED: Deep Emotion Signals (DES) Attention Module
                    # This simulates the 'X' factor by allowing the model to focus 
                    # on latent emotional cues in the sequence before pooling.
                    self.des_attention = nn.MultiheadAttention(
                        embed_dim=social_dim, 
                        num_heads=heads, 
                        dropout=drop, 
                        batch_first=True
                    )
                    self.des_norm = nn.LayerNorm(social_dim)
                    
                    # Freeze early layers as you did before
                    if hasattr(self.encoder_social, "encoder"):
                        for i, layer in enumerate(self.encoder_social.encoder.layer):
                            if i < freeze_first_n_layers:
                                for p in layer.parameters():
                                    p.requires_grad = False
                    
                    print(f"      ✓ Social stream active ({social_dim}D) + DES Module")
                except Exception as e:
                    print(f"      [WARN] Failed to load social model: {e}")
                    self.use_social_stream = False
                    social_dim = 0
            else:
                self.encoder_social = None
                social_dim = 0
            
            # (Rest of your freezing/init code for encoder_base remains the same...)
    
            # --- FUSION GATE (Enhanced for X-Logic) ---
            if self.use_social_stream:
                fusion_input = base_dim + social_dim
                self.fusion_gate = nn.Sequential(
                    nn.Linear(fusion_input, dim * 2), # Project higher first
                    nn.GELU(),
                    nn.Linear(dim * 2, dim),         # Then compress to GNN dim
                    nn.LayerNorm(dim),
                    nn.Dropout(drop),
                )
                print(f"      ✓ Fusion: {base_dim} + {social_dim} → {dim}")
            else:
                self.fusion_gate = nn.Sequential(
                    nn.Linear(base_dim, dim),
                    nn.LayerNorm(dim),
                    nn.GELU(),
                    nn.Dropout(drop),
                )
            
            # Init projection (no dropout)
            self.init_proj = nn.Sequential(
                nn.Linear(base_dim, dim),
                nn.LayerNorm(dim),
                nn.GELU(),
            )
        
            # Emotion embeddings
            self.emb = nn.Parameter(torch.randn(n_emo, dim) * 0.02)
            
            # Graph layers
            self.graph = nn.ModuleList([
                AdaptiveGraphLayer(dim, heads=heads, max_k=max_k, drop=drop)
                for _ in range(n_layers)
            ])
            
            self.cross_attn = nn.MultiheadAttention(dim, heads, dropout=drop, batch_first=True)
            
            # Classifiers
            self.cls_28 = nn.Sequential(
                nn.Linear(dim * 2, 512),
                nn.LayerNorm(512),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(512, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(256, n_emo),
            )
            
            self.cls_ek = nn.Sequential(
                nn.Linear(dim * 2, 256),
                nn.LayerNorm(256),
                nn.GELU(),
                nn.Dropout(drop),
                nn.Linear(256, n_ek),
            )
            
            self.sentic = sentic
            self.knowledge_warmup_steps = knowledge_warmup_steps
            self.prior_scale = prior_scale
            self.global_step = 0
            
            # SenticNet affective residual
            if self.sentic is not None:
                label_aff = self.sentic.get_label_affective_scores()
                self.register_buffer("label_aff", label_aff)
                self.sentic_res_proj = nn.Sequential(
                    nn.Linear(4, dim * 2),
                    nn.GELU(),
                    nn.Dropout(drop),
                )
            else:
                self.label_aff = None
                self.sentic_res_proj = None
            
            # Initialize from label texts
            if init_from_label_texts and tokenizer and emotion_names:
                self._init_emotion_nodes(tokenizer, emotion_names, n_emo)
    
    def _init_emotion_nodes(self, tokenizer, emotion_names, n_emo):
        was_training = self.encoder_base.training
        self.encoder_base.eval()
        
        with torch.no_grad():
            batch = tokenizer(
                emotion_names[:n_emo],
                padding=True,
                truncation=True,
                max_length=8,
                return_tensors="pt"
            )
            out = self.encoder_base(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
            ).last_hidden_state
            
            m = batch["attention_mask"].unsqueeze(-1).float()
            pooled = (out * m).sum(1) / m.sum(1).clamp(min=1e-9)
            
            init_nodes = self.init_proj(pooled)
            self.emb.data.copy_(init_nodes.contiguous())
        
        self.encoder_base.train(was_training)
        print("      ✓ Emotion nodes initialized from label texts")
    
    def forward(self, ids, mask, prior_bias=None, affect_vec=None):
        base_out = self.encoder_base(ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).float()
        base_pooled = (base_out * m).sum(1) / m.sum(1).clamp(min=1e-9)
        
        if self.use_social_stream and self.encoder_social:
            social_hidden = self.encoder_social(ids, attention_mask=mask).last_hidden_state
        
            attn_out, _ = self.des_attention(social_hidden, social_hidden, social_hidden)
            social_refined = self.des_norm(social_hidden + attn_out)
            

            social_pooled = social_refined[:, 0, :] 
            
 
            combined = torch.cat([base_pooled, social_pooled], dim=-1)
            text_p = self.fusion_gate(combined)
        else:
            text_p = self.fusion_gate(base_pooled)
        

        nodes = self.emb
        

        progress = min(1.0, self.global_step / max(1, self.knowledge_warmup_steps * 2))
        current_k = int(28 - progress * 23)
        
        for gl in self.graph:
            nodes = gl(nodes, current_k=current_k)

        q = text_p.unsqueeze(1)
        k = nodes.unsqueeze(0).expand(q.size(0), -1, -1)
        att_emo, _ = self.cross_attn(q, k, k)
        att_emo = att_emo.squeeze(1)
        
        comb = torch.cat([text_p, att_emo], dim=-1)
        

        if affect_vec is not None and self.sentic_res_proj:
            comb = comb + self.sentic_res_proj(affect_vec)
        
        l28 = self.cls_28(comb)
        lek = self.cls_ek(comb)
        
 
        if prior_bias is not None:
            warm = min(1.0, self.global_step / float(max(1, self.knowledge_warmup_steps)))
            l28 = l28 + (self.prior_scale * warm) * prior_bias
        
        return l28, lek

In [11]:
def compute_pos_weights(ds_train, n_labels=7, power=0.5, device="cpu"):
    # ds_train is now a Dataset object with pre-converted 7-class labels
    counts = np.zeros(n_labels, dtype=np.float64)
    for item in ds_train:
        counts += item["labels"].numpy()
    w = np.power(counts.max() / (counts + 1.0), power)
    w = np.clip(w, 1.0, 50.0)
    return torch.FloatTensor(w).to(device)

def train_epoch(model, loader, opt, sched, dev, pos_wts, emo_to_ek, alpha=0.2, epoch=0):
    model.train()
    tl = tb28 = tbek = 0.0
    
    scaler = torch.amp.GradScaler("cuda") if dev.type == "cuda" else None
    alpha_current = alpha * min(1.0, (epoch + 1) / 3)
    
    for batch in tqdm(loader, desc=f"Epoch {epoch+1}", leave=False):
        ids = batch["input_ids"].to(dev)
        mask = batch["attention_mask"].to(dev)
        l28 = batch["labels"].to(dev)
        lek = make_ekman_targets(l28, emo_to_ek)
        
        opt.zero_grad(set_to_none=True)
        
        prior_bias = None
        affect_vec = None
        if model.sentic:
            priors = [model.sentic.compute_label_prior(t) for t in batch["text"]]
            prior_bias = torch.tensor(np.stack(priors), dtype=torch.float32, device=dev)
            if model.label_aff is not None:
                affect_vec = prior_bias @ model.label_aff.to(dev)
        
        if scaler:
            with torch.amp.autocast("cuda"):
                o28, oek = model(ids, mask, prior_bias, affect_vec)
                b28 = F.binary_cross_entropy_with_logits(o28, l28, pos_weight=pos_wts)
                bek = F.binary_cross_entropy_with_logits(oek, lek)
                loss = b28 + alpha_current * bek
            
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
        else:
            o28, oek = model(ids, mask, prior_bias, affect_vec)
            b28 = F.binary_cross_entropy_with_logits(o28, l28, pos_weight=pos_wts)
            bek = F.binary_cross_entropy_with_logits(oek, lek)
            loss = b28 + alpha_current * bek
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        
        if sched:
            sched.step()
        
        model.global_step += 1
        
        tl += loss.item()
        tb28 += b28.item()
        tbek += bek.item()
    
    n = len(loader)
    return {"total": tl/n, "bce28": tb28/n, "bceek": tbek/n, "alpha": alpha_current}


In [16]:
@torch.no_grad()
def predict_probs(model, loader, dev):
    model.eval()
    probs_list, labels_list = [], []
    
    for batch in tqdm(loader, desc="Eval", leave=False):
        ids = batch["input_ids"].to(dev)
        mask = batch["attention_mask"].to(dev)
        
        prior_bias = None
        affect_vec = None
        if model.sentic:
            priors = [model.sentic.compute_label_prior(t) for t in batch["text"]]
            prior_bias = torch.tensor(np.stack(priors), dtype=torch.float32, device=dev)
            if model.label_aff is not None:
                affect_vec = prior_bias @ model.label_aff.to(dev)
        
        o28, _ = model(ids, mask, prior_bias, affect_vec)
        probs_list.append(torch.sigmoid(o28).cpu())
        labels_list.append(batch["labels"])
    
    return torch.cat(probs_list).numpy(), torch.cat(labels_list).numpy()

def find_thresholds(probs, labels):
    thrs = []
    for i in range(probs.shape[1]):  # was range(28)
        best_f1, best_t = 0.0, 0.5
        for t in np.arange(0.05, 0.95, 0.025):
            f1 = f1_score(labels[:, i], (probs[:, i] > t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thrs.append(best_t)
    return np.array(thrs)

In [17]:
def main():
    CFG = {
        "base_model": "roberta-base",
        "social_model": "cardiffnlp/twitter-roberta-base-emotion",
        "use_social_stream": True,
        "bs": 24,
        "lr": 2e-5,
        "epochs": 7,
        "max_len": 96,
        "dim": 384,
        "layers": 2,
        "heads": 4,
        "max_k": 10,
        "drop": 0.1,
        "alpha_ekman": 0.2,
        "warmup_ratio": 0.1,
        "pos_weight_power": 0.5,
        "freeze_first_n_layers": 2,
        "knowledge_warmup_steps": 1000,
        "prior_scale": 0.30,
        "patience": 3,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }
    
    
    dev = torch.device(CFG["device"])
    
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("\nLoading dataset...")
    try:
        ds = load_dataset("cirimus/super-emotion")
    except:
        cache = os.path.expanduser("~/.cache/huggingface/datasets/google-research-datasets___go_emotions")
        shutil.rmtree(cache, ignore_errors=True)
        ds = load_dataset("google-research-datasets/go_emotions", "simplified")
    
    emotion_names = EKMAN_CLASSES   # just the 7 Ekman names
    emo_to_ek = {i: i for i in range(7)}
    
    tokenizer = AutoTokenizer.from_pretrained(CFG["base_model"], use_fast=False)
    
    print("\nInitializing SenticNet...")
    # NEW — same line, emotion_names is now 7 classes, SenticNet handles it automatically
    sentic = RealSenticNetPrior(EKMAN_CLASSES, bfs_depth=3, max_concepts_per_text=32)
    if not (sentic.sn or sentic.fallback):
        print("  → Running without SenticNet")
        sentic = None
    
    train_ds = GoEmotionsEkman7Dataset(ds["train"],      tokenizer, CFG["max_len"])
    val_ds   = GoEmotionsEkman7Dataset(ds["validation"], tokenizer, CFG["max_len"])
    test_ds  = GoEmotionsEkman7Dataset(ds["test"],       tokenizer, CFG["max_len"])
    
    train_loader = DataLoader(train_ds, CFG["bs"], shuffle=True, num_workers=0, pin_memory=False)
    val_loader = DataLoader(val_ds, CFG["bs"]*2, shuffle=False, num_workers=0, pin_memory=False)
    test_loader = DataLoader(test_ds, CFG["bs"]*2, shuffle=False, num_workers=0, pin_memory=False)
    
    pos_wts = compute_pos_weights(train_ds, n_labels=7, power=CFG["pos_weight_power"], device=dev)
    print(f"Pos-weights: {pos_wts.min():.2f} - {pos_wts.max():.2f}")
    
    print("\n" + "="*80)
    print("BUILDING DUAL-STREAM MODEL")
    print("="*80)
    
    model = HybridSenticGNN(
        base_model_name=CFG["base_model"],
        social_model_name=CFG["social_model"],
        emotion_names=EKMAN_CLASSES,
        tokenizer=tokenizer,
        n_emo=7,
        n_ek=7,
        dim=CFG["dim"],
        n_layers=CFG["layers"],
        heads=CFG["heads"],
        max_k=CFG["max_k"],
        drop=CFG["drop"],
        sentic=sentic,
        freeze_first_n_layers=CFG["freeze_first_n_layers"],
        knowledge_warmup_steps=CFG["knowledge_warmup_steps"],
        prior_scale=CFG["prior_scale"],
        init_from_label_texts=True,
        use_social_stream=CFG["use_social_stream"],
    ).to(dev)
    
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nParams: {total:,} | Trainable: {trainable:,}")
    
    if model.use_social_stream:
        print("DUAL-STREAM ACTIVE: Formal + Social encoders")
    else:
        print("SINGLE-STREAM: Using base only")
    
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    
    steps = len(train_loader) * CFG["epochs"]
    warmup = int(steps * CFG["warmup_ratio"])
    sched = get_cosine_schedule_with_warmup(opt, warmup, steps)
    
    print(f"Steps: {steps}, Warmup: {warmup}")
    
    print("\n" + "="*80)
    print("TRAINING")
    print("="*80)
    
    best_f1 = -1.0
    patience = 0
    
    for ep in range(CFG["epochs"]):
        print(f"\nEpoch {ep+1}/{CFG['epochs']}")
        losses = train_epoch(model, train_loader, opt, sched, dev, pos_wts, emo_to_ek, CFG["alpha_ekman"], ep)
        print(f"Loss: {losses['total']:.4f} | BCE28: {losses['bce28']:.4f} | BCEEk: {losses['bceek']:.4f}")
        
        vp, vl = predict_probs(model, val_loader, dev)
        vmacro = f1_score(vl, (vp>0.5).astype(int), average="macro", zero_division=0)
        print(f"Val Macro-F1: {vmacro:.4f}")
        
        if vmacro > best_f1:
            best_f1 = vmacro
            patience = 0
            torch.save(model.state_dict(), "ekman7_dual_stream.pt")
            print(f"✓ Saved! Best: {best_f1:.4f}")
        else:
            patience += 1
            if patience >= CFG["patience"]:
                print("Early stopping")
                break
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    print("\n" + "="*80)
    print("FINAL EVALUATION")
    print("="*80)
    
    model.load_state_dict(torch.load("ekman7_dual_stream.pt", map_location=dev))
    
    vp, vl = predict_probs(model, val_loader, dev)
    thrs = find_thresholds(vp, vl)
    print(f"Thresholds: min={thrs.min():.3f}, max={thrs.max():.3f}, mean={thrs.mean():.3f}")
    
    tp, tl = predict_probs(model, test_loader, dev)
    tpred = np.zeros_like(tp)
    for i in range(7):
        tpred[:, i] = (tp[:, i] > thrs[i]).astype(int)
    
    macro = f1_score(tl, tpred, average="macro", zero_division=0)
    micro = f1_score(tl, tpred, average="micro", zero_division=0)
    
    print("\n" + "="*80)
    print("FINAL RESULTS (DUAL-STREAM HYBRID)")
    print("="*80)
    print(f"Macro F1: {macro:.4f}")
    print(f"Micro F1: {micro:.4f}")
    print("="*80)
    
    per_class = f1_score(tl, tpred, average=None, zero_division=0)
    print("\nPer-class F1:")
    for name, score in zip(emotion_names, per_class):
        print(f"  {name:20s}: {score:.4f}")
    
    print(f"\n Features:")
    print(f"  Dual-Stream: {'YES' if model.use_social_stream else 'NO'}")
    print(f"  SenticNet: {'YES' if sentic else 'NO'}")
    print(f"  Affective Residual: {'YES' if model.label_aff is not None else 'NO'}")
    print(f"  Label Init: YES")
    print(f"  Ekman Aux: YES")
    
    weak = [(n, s) for n, s in zip(emotion_names, per_class) if s < 0.40]
    if weak:
        print(f"\n Weak classes ({len(weak)}/7):")
        for n, s in sorted(weak, key=lambda x: x[1])[:5]:
            print(f"  {n:20s}: {s:.4f}")
    
    print(f"\nCheckpoint: ekman7_dual_stream.pt")
    print(f" Viz: {os.path.abspath(VIZ_DIR)}")
    return macro


if __name__ == "__main__":
    final = main()
    print(f"\n{'='*80}")
    print(f"FINAL MACRO F1: {final:.4f}")
    print(f"{'='*80}")


Loading dataset...

Initializing SenticNet...
[INFO] ✓ SenticNet loaded
Pre-tokenizing...
Pre-tokenizing...
Pre-tokenizing...
Pos-weights: 1.00 - 4.89

BUILDING DUAL-STREAM MODEL
[1/2] Loading base encoder: roberta-base
[2/2] Loading social encoder: cardiffnlp/twitter-roberta-base-emotion
      ✓ Social stream active (768D) + DES Module
      ✓ Fusion: 768 + 768 → 384
      ✓ Emotion nodes initialized from label texts

Params: 257,120,910 | Trainable: 242,945,166
DUAL-STREAM ACTIVE: Formal + Social encoders
Steps: 12663, Warmup: 1266

TRAINING

Epoch 1/7


Loss: 0.4027 | BCE28: 0.3816 | BCEEk: 0.3170


Val Macro-F1: 0.6073
✓ Saved! Best: 0.6073

Epoch 2/7


Loss: 0.3106 | BCE28: 0.2822 | BCEEk: 0.2125


Val Macro-F1: 0.6334
✓ Saved! Best: 0.6334

Epoch 3/7


...

Val Macro-F1: 0.6148

Epoch 6/7


Loss: 0.1595 | BCE28: 0.1380 | BCEEk: 0.1078


Val Macro-F1: 0.6118
Early stopping

FINAL EVALUATION


Thresholds: min=0.250, max=0.600, mean=0.439



FINAL RESULTS (DUAL-STREAM HYBRID)
Macro F1: 0.6413
Micro F1: 0.7066

Per-class F1:
  anger               : 0.5836
  disgust             : 0.5045
  fear                : 0.6550
  joy                 : 0.8297
  sadness             : 0.6058
  surprise            : 0.6198
  neutral             : 0.6910

 Features:
  Dual-Stream: YES
  SenticNet: YES
  Affective Residual: YES
  Label Init: YES
  Ekman Aux: YES

✅ Checkpoint: ekman7_dual_stream.pt
✅ Viz: /home/jovyan/viz_outputs

FINAL MACRO F1: 0.6413
